In [102]:
import cv2
import numpy as np
from math import log
from PIL import Image
from typing import Tuple


In [103]:
def hue_calc(delta: float, C_max: float, R_1: float, G_1: float, B_1: float) -> float:
    if delta == 0:
        return 0.0
    elif C_max == R_1:
        return 60 * (((G_1 - B_1) / delta) % 6)
    elif C_max == G_1:
        return 60 * (((B_1 - R_1) / delta) + 2)
    elif C_max == B_1:
        return 60 * (((R_1 - G_1) / delta) + 4)

In [104]:
def sat_calc_hls(C_max: float, delta: float, L: float) -> float:
    if delta == 0:
        return 0.0
    return delta / (1 - abs(2 * L - 1))

In [105]:
def rgb_to_hls(R: float, G: float, B: float, scale_coef_hue: float, scale_coef_ls: float) -> Tuple[float, float, float]:
    R_1 = R / 255
    G_1 = G / 255
    B_1 = B / 255

    C_max = max(R_1, G_1, B_1)
    C_min = min(R_1, G_1, B_1)

    delta = C_max - C_min

    L = (C_max + C_min) / 2

    H = hue_calc(delta, C_max, R_1, G_1, B_1)

    S = sat_calc_hls(C_max, delta, L)

    H = H * scale_coef_hue
    L = L * scale_coef_ls
    S = S * scale_coef_ls
    
    return H, L, S

In [106]:
def hls_to_rgb(H: float, L: float, S: float) -> Tuple[float, float, float]:
    C = (1 - abs(2 * L - 1)) * S
    X = C * (1 - abs((H / 60) % 2 - 1))
    m = L - C / 2

    R_1, G_1, B_1 = pre_colors_calc(C, X, H)

    R = (R_1 + m) * 255
    G = (G_1 + m) * 255
    B = (B_1 + m) * 255

    return R, G, B

In [107]:
def pre_colors_calc(C: float, X: float, H: float) -> Tuple[float, float, float]:
    if H >= 360:
        H = H % 360
    if (0 <= H and H < 60):
        return C, X, 0.0
    elif (60 <= H and H < 120):
        return X, C, 0.0
    elif (120 <= H and H < 180):
        return 0.0, C, X
    elif (180 <= H and H < 240):
        return 0.0, X, C
    elif (240 <= H and H < 300):
        return X, 0.0, C
    elif (300 <= H and H < 360):
        return C, 0.0, X

In [108]:
def change_temp(T: float) -> Tuple[float, float, float]:
    r, g, b = 0, 0, 0
    norm_T = T / 100

    if norm_T < 67:
        r = 255
        g = -155.25485562709179 - 0.44596950469579133 * (norm_T - 2) + 104.49216199393888 * np.log(norm_T - 2)
        if norm_T < 20:
            b = 0
        else:
            b = -254.76935184120902 + 0.8274096064007395 * (norm_T - 10) + 115.67994401066147 * np.log(norm_T - 10)
    else:
        r = 351.97690566805693 + 0.114206453784165 * (norm_T - 55) - 40.25366309332127 * np.log(norm_T - 55)
        g = 325.4494125711974 + 0.07943456536662342 * (norm_T - 50) - 28.0852963507957 * np.log(norm_T - 50)
        b = 255

    r = max(0, min(r, 255))
    g = max(0, min(g, 255))
    b = max(0, min(b, 255))

    return r, g, b

In [109]:
print("R, G, B")
print(change_temp(6000))

R, G, B
(255, np.float64(243.16338192572562), np.float64(239.14373071517548))


In [110]:
def compute_temperature(image_path: str) -> float:
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    R, G, B = image[:, :, 0], image[:, :, 1], image[:, :, 2]

    R_mean = np.mean(R)
    B_mean = np.mean(B)
    
    ratio = R_mean / B_mean

    if ratio > 0.64:
        T = -459.87 + 437 * ratio
    elif 0.38 <= ratio <= 0.64:
        T = -459.87 + 3448.2 * ratio - 0.2467
    else:
        T = -459.87 + 2220.8 * (ratio ** 2) + 284.16
    
    return T

## 1

In [111]:
print("Температура изображения (Кельвины):", round(compute_temperature("origins/mononoke.jpg")))

Температура изображения (Кельвины): -117


In [112]:
T = 10000
alpha = 0.3

In [113]:
image = cv2.imread("origins/mononoke.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

changed_temp_array = np.zeros_like(image, dtype=np.uint8)

r, g, b = change_temp(T)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        R, G, B = map(float, image[i, j])

        Lum = (min(R, G, B) + max(R, G, B)) / 2

        R = (1 - alpha) * R + alpha * r
        R = max(0, min(R, 255))

        G = (1 - alpha) * G + alpha * g
        G = max(0, min(G, 255))

        B = (1 - alpha) * B + alpha * b
        B = max(0, min(B, 255))

        H, L, S = rgb_to_hls(R, G, B, 1, 1)

        L = Lum / 255

        R, G, B = hls_to_rgb(H, L, S)

        changed_temp_array[i, j] = (R, G, B)

changed_temp_array = cv2.cvtColor(changed_temp_array, cv2.COLOR_RGB2BGR)

cv2.imwrite(f"results/change_temp/mononoke_changed_temp_{T}.png", changed_temp_array)

True

## 2

In [120]:
print("Температура изображения (Кельвины):", round(compute_temperature("origins/rengoku.jpg")))

Температура изображения (Кельвины): 1334


In [124]:
print("Температура изображения (Кельвины):", round(compute_temperature("results/change_temp/rengoku_changed_temp_1000.png")))

Температура изображения (Кельвины): 1562


In [ ]:
T = 1000
alpha = 0.3

In [122]:
image = cv2.imread("origins/rengoku.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

changed_temp_array = np.zeros_like(image, dtype=np.uint8)

r, g, b = change_temp(T)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        R, G, B = map(float, image[i, j])

        Lum = (min(R, G, B) + max(R, G, B)) / 2

        R = (1 - alpha) * R + alpha * r
        R = max(0, min(R, 255))

        G = (1 - alpha) * G + alpha * g
        G = max(0, min(G, 255))

        B = (1 - alpha) * B + alpha * b
        B = max(0, min(B, 255))

        H, L, S = rgb_to_hls(R, G, B, 1, 1)

        L = Lum / 255

        R, G, B = hls_to_rgb(H, L, S)

        changed_temp_array[i, j] = (R, G, B)

changed_temp_array = cv2.cvtColor(changed_temp_array, cv2.COLOR_RGB2BGR)

cv2.imwrite(f"results/change_temp/rengoku_changed_temp_{T}.png", changed_temp_array)

True

## 3

In [117]:
print("Температура изображения (Кельвины):", round(compute_temperature("origins/rukia.jpg")))

Температура изображения (Кельвины): -149


In [118]:
T = 10000
alpha = 0.3

In [119]:
image = cv2.imread("origins/rukia.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

changed_temp_array = np.zeros_like(image, dtype=np.uint8)

r, g, b = change_temp(T)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        R, G, B = map(float, image[i, j])

        Lum = (min(R, G, B) + max(R, G, B)) / 2

        R = (1 - alpha) * R + alpha * r
        R = max(0, min(R, 255))

        G = (1 - alpha) * G + alpha * g
        G = max(0, min(G, 255))

        B = (1 - alpha) * B + alpha * b
        B = max(0, min(B, 255))

        H, L, S = rgb_to_hls(R, G, B, 1, 1)

        L = Lum / 255

        R, G, B = hls_to_rgb(H, L, S)

        changed_temp_array[i, j] = (R, G, B)

changed_temp_array = cv2.cvtColor(changed_temp_array, cv2.COLOR_RGB2BGR)

cv2.imwrite(f"results/change_temp/rukia_changed_temp_{T}.png", changed_temp_array)

True